In [32]:
import numpy as np
import pandas as pd
import zipfile
import unicodedata
import pandas as pd
from sklearn.model_selection import train_test_split
import re
import html


In [33]:

def load_spamassassin():
    """
    Loads spam_assassin.csv directly, or from spam_assassin.csv.zip if needed.
    Expected columns:
        - text
        - target
    """
    try:
        return pd.read_csv("datasets/spam_assassin.csv")
    except FileNotFoundError:
        with zipfile.ZipFile("spam_assassin.csv.zip", "r") as z:
            csv_files = [f for f in z.namelist() if f.endswith(".csv")]
            if not csv_files:
                raise FileNotFoundError("No CSV file found inside spam_assassin.csv.zip")
            with z.open(csv_files[0]) as f:
                return pd.read_csv(f)


In [34]:
def strip_email_headers(text):
    """
    Removes common email header / transport lines.
    """
    if pd.isna(text):
        return ""

    header_prefixes = [
        "from:",
        "to:",
        "cc:",
        "bcc:",
        "subject:",
        "date:",
        "reply-to:",
        "return-path:",
        "received:",
        "message-id:",
        "mime-version:",
        "content-type:",
        "content-transfer-encoding:",
        "x-mailer:",
        "x-spam-",
        "delivered-to:",
        "errors-to:",
        "sender:",
        "in-reply-to:",
        "references:",
        "mailing-list:",
        "precedence:",
        "list-id:",
        "list-unsubscribe:",
        "list-post:",
        "list-help:",
        "list-subscribe:",
        "return-receipt-to:",
        "delivery-date:",
        "status:",
        "path:",
        "x-keywords:",
    ]

    cleaned_lines = []

    for line in str(text).splitlines():
        line_lower = line.strip().lower()

        if any(line_lower.startswith(prefix) for prefix in header_prefixes):
            continue

        # remove common mbox-style first line
        if line_lower.startswith("from ") and re.search(r"\b\d{4}\b", line_lower):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)



In [35]:

def normalize_urls(text):
    return re.sub(r"(https?://\S+|www\.\S+)", " <URL> ", text)


def normalize_emails(text):
    return re.sub(
        r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}\b",
        " <EMAIL> ",
        text
    )


def normalize_numbers(text):
    return re.sub(r"\b\d+(?:[\.,:/-]\d+)*\b", " <NUM> ", text)


def remove_html_tags(text):
    return re.sub(r"<[^>]+>", " ", text)

In [36]:

def clean_text(text, remove_headers=False):
    """
    General cleaner for SpamAssassin text.
    """
    if pd.isna(text):
        return ""

    text = str(text)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)

    if remove_headers:
        text = strip_email_headers(text)

    text = remove_html_tags(text)
    text = normalize_urls(text)
    text = normalize_emails(text)
    text = normalize_numbers(text)

    # remove control chars
    text = re.sub(r"[\r\t]", " ", text)
    text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)

    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()

    return text




In [49]:
def preprocess_spam_dataset(test_size=0.2):
    """
    Creates two cleaned versions:
        1. text_clean_keep_headers
        2. text_clean_no_headers

    Also drops duplicate instances based on text_clean_no_headers,
    which is the stricter and more general deduplication choice.
    """
    
    

    df = load_spamassassin()
    
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

    df["text_clean_keep_headers"] = df["text"].apply(
        lambda x: clean_text(x, remove_headers=False)
    )

    df["text_clean_no_headers"] = df["text"].apply(
        lambda x: clean_text(x, remove_headers=True)
    )





    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=42,
        stratify=df["target"]
    )

    return train_df, test_df, df

In [38]:
spam_assassin = load_spamassassin()

In [39]:
spam_assassin.info()

<class 'pandas.DataFrame'>
RangeIndex: 5796 entries, 0 to 5795
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    5796 non-null   str  
 1   target  5796 non-null   int64
dtypes: int64(1), str(1)
memory usage: 23.3 MB


In [40]:
spam_assassin['target'].head()

0    0
1    1
2    1
3    1
4    0
Name: target, dtype: int64

In [41]:
spam_assassin['target'].value_counts(normalize=True)

target
0    0.672878
1    0.327122
Name: proportion, dtype: float64

In [42]:
spam_assassin['text'].duplicated().value_counts()

text
False    5329
True      467
Name: count, dtype: int64

In [43]:
print(spam_assassin['text'][1])

From gort44@excite.com Mon Jun 24 17:54:21 2002 Return-Path: gort44@excite.com Delivery-Date: Tue Jun 4 05:31:16 2002 Received: from mandark.labs.netnoteinc.com ([213.105.180.140]) by dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id g544VFO20182 for <jm@jmason.org>; Tue, 4 Jun 2002 05:31:15 +0100 Received: from wi-poli.poli.cl ([200.54.149.34]) by mandark.labs.netnoteinc.com (8.11.2/8.11.2) with SMTP id g544VC729935; Tue, 4 Jun 2002 05:31:13 +0100 Received: from 216.77.61.89 (unverified [218.5.180.148]) by wi-poli.poli.cl (EMWAC SMTPRS 0.83) with SMTP id <B0000918901@wi-poli.poli.cl>; Tue, 04 Jun 2002 00:14:29 -0400 Message-Id: <B0000918901@wi-poli.poli.cl> To: <chrbader@telecom.at> From: "irese" <gort44@excite.com> Subject: Cash in on your home equity Date: Tue, 04 Jun 2002 00:18:34 -1600 MIME-Version: 1.0 Content-Type: text/plain; charset="Windows-1252" X-Keywords: Content-Transfer-Encoding: 7bit Mortgage Lenders & Brokers Are Ready to compete for your business. Whether a new home l

In [44]:
spam_assassin['target'].value_counts()

target
0    3900
1    1896
Name: count, dtype: int64

In [50]:
train_df, test_df, full_df = preprocess_spam_dataset()

In [53]:
print("\nTarget distribution:")
print(full_df["target"].value_counts(normalize=True))


Target distribution:
target
0    0.68268
1    0.31732
Name: proportion, dtype: float64


In [55]:
for i in range(10):
    print(full_df["text"][i] + "\n\n")

From ilug-admin@linux.ie Mon Jul 29 11:28:02 2002 Return-Path: <ilug-admin@linux.ie> Delivered-To: yyyy@localhost.netnoteinc.com Received: from localhost (localhost [127.0.0.1]) by phobos.labs.netnoteinc.com (Postfix) with ESMTP id A13D94414F for <jm@localhost>; Mon, 29 Jul 2002 06:25:11 -0400 (EDT) Received: from phobos [127.0.0.1] by localhost with IMAP (fetchmail-5.9.0) for jm@localhost (single-drop); Mon, 29 Jul 2002 11:25:11 +0100 (IST) Received: from lugh.tuatha.org (root@lugh.tuatha.org [194.125.145.45]) by dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id g6RHn7i17130 for <jm-ilug@jmason.org>; Sat, 27 Jul 2002 18:49:07 +0100 Received: from lugh (root@localhost [127.0.0.1]) by lugh.tuatha.org (8.9.3/8.9.3) with ESMTP id SAA25016; Sat, 27 Jul 2002 18:45:03 +0100 X-Authentication-Warning: lugh.tuatha.org: Host root@localhost [127.0.0.1] claimed to be lugh Received: from mail1.mail.iol.ie (mail1.mail.iol.ie [194.125.2.192]) by lugh.tuatha.org (8.9.3/8.9.3) with ESMTP id SAA24977 fo

In [56]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [57]:
def the_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
        
        ( "clf", LogisticRegression(max_iter=2000, random_state=42, class_weight="balanced"))
    ])


In [58]:
def evaluate_model(model_name, model, X_test, y_test):
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    print(f"\n{'=' * 25} {model_name} {'=' * 25}")
    print(classification_report(y_test, predictions, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, predictions))
    print(f"ROC AUC: {roc_auc_score(y_test, probabilities):.4f}")

    report = classification_report(y_test, predictions, zero_division=0, output_dict=True)

    summary = {
        "model": model_name,
        "accuracy": report["accuracy"],
        "macro_f1": report["macro avg"]["f1-score"],
        "spam_precision": report["1"]["precision"],
        "spam_recall": report["1"]["recall"],
        "spam_f1": report["1"]["f1-score"],
        "roc_auc": roc_auc_score(y_test, probabilities),
    }
    return summary


In [61]:
# use the same split for both text versions
indices = np.arange(len(full_df))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=full_df["target"]
)

train_df = full_df.iloc[train_idx].reset_index(drop=True)
test_df = full_df.iloc[test_idx].reset_index(drop=True)

y_train = train_df["target"].values
y_test = test_df["target"].values

summaries = []

# Model 1: keep headers
X_train_keep = train_df["text_clean_keep_headers"].fillna("").values
X_test_keep = test_df["text_clean_keep_headers"].fillna("").values

model_keep = the_pipeline()
model_keep.fit(X_train_keep, y_train)

summaries.append(
    evaluate_model(
        "KEEP HEADERS",
        model_keep,
        X_test_keep,
        y_test
    )
)

# Model 2: remove headers
X_train_no = train_df["text_clean_no_headers"].fillna("").values
X_test_no = test_df["text_clean_no_headers"].fillna("").values

model_no = the_pipeline()
model_no.fit(X_train_no, y_train)

summaries.append(
    evaluate_model(
        "NO HEADERS",
        model_no,
        X_test_no,
        y_test
    )
)

summary_df = pd.DataFrame(summaries)

print(f"\n{'=' * 22} SUMMARY COMPARISON {'=' * 22}")
print(summary_df.to_string(index=False))


========================= KEEP HEADERS =========================
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       728
           1       1.00      0.98      0.99       338

    accuracy                           0.99      1066
   macro avg       0.99      0.99      0.99      1066
weighted avg       0.99      0.99      0.99      1066

Confusion Matrix:
[[727   1]
 [  6 332]]
ROC AUC: 0.9996

========================= NO HEADERS =========================
              precision    recall  f1-score   support

           0       0.69      1.00      0.82       728
           1       1.00      0.03      0.06       338

    accuracy                           0.69      1066
   macro avg       0.85      0.52      0.44      1066
weighted avg       0.79      0.69      0.58      1066

Confusion Matrix:
[[728   0]
 [327  11]]
ROC AUC: 0.5163

====================== SUMMARY COMPARISON ======================
       model  accuracy  macro_f1  sp

In [62]:
print(full_df["text_clean_keep_headers"].str.len().describe())
print(full_df["text_clean_no_headers"].str.len().describe())
print((full_df["text_clean_no_headers"].str.strip() == "").sum())

count      5329.000000
mean       3075.675361
std        5560.337949
min         301.000000
25%        1745.000000
50%        2419.000000
75%        3246.000000
max      231926.000000
Name: text_clean_keep_headers, dtype: float64
count    5329.000000
mean        8.458623
std       151.688154
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max      7648.000000
Name: text_clean_no_headers, dtype: float64
5299
